<a href="https://colab.research.google.com/github/Bindiya-Anand/StreamFlix-Content-Analytics-Project/blob/main/notebooks/StreamFlix_Phase3_KPI_Bindiya_Anand.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Internmo Data Analytics Project
## StreamFlix Content Analysis
### Data Analyst Intern: Bindiya Anand
#### Intern Id: Inter563

# Phase 3: KPI Calculations

## Import libraries and load data

In [3]:
import pandas as pd
import numpy as np

#ignore filter warnings
import warnings
warnings.filterwarnings('ignore')

# Show numbers with 2 decimals
pd.options.display.float_format = '{:,.2f}'.format

In [4]:
subs = pd.read_csv('subscribers.csv')
titles = pd.read_csv('titles.csv')
watch = pd.read_csv('watch_history.csv')
watchlist = pd.read_csv('watchlist.csv')

print('subscribers :', subs.shape)
print('titles      :', titles.shape)
print('watch       :', watch.shape)
print('watchlist   :', watchlist.shape)

subscribers : (15000, 14)
titles      : (9000, 21)
watch       : (443786, 10)
watchlist   : (65000, 5)


In [5]:
# quick data prep
# Turn True/False columns into real booleans (works even if they were read as text)
subs['is_active'] = subs['is_active'].astype(str).str.lower() == 'true'
titles['is_original'] = titles['is_original'].astype(str).str.lower() == 'true'
watchlist['watched'] = watchlist['watched'].astype(str).str.lower() == 'true'

# Turn date text into real dates
subs['signup_date'] = pd.to_datetime(subs['signup_date'])
subs['churn_date'] = pd.to_datetime(subs['churn_date'])
watch['watch_date'] = pd.to_datetime(watch['watch_date'])

subs.head()

,subscriber_id,signup_date,country,region,age,gender,plan_type,monthly_price_usd,household_size,primary_device,payment_method,tenure_months,is_active,churn_date
0,SUB100000,2023-01-09,France,Europe,22,Female,Basic with Ads,6.99,1,Mobile,Credit Card,41,True,NaT
1,SUB100001,2020-02-05,United Kingdom,Europe,48,Male,Basic with Ads,6.99,1,Smart TV,Credit Card,76,False,2022-12-08
2,SUB100002,2022-04-15,Italy,Europe,35,Male,Standard,15.49,2,Streaming Stick,Gift Card,50,True,NaT
3,SUB100003,2024-10-10,Spain,Europe,49,Female,Basic with Ads,6.99,1,Laptop,Credit Card,19,True,NaT
4,SUB100004,2020-04-12,United States,North America,36,Female,Basic with Ads,6.99,1,Mobile,Credit Card,74,False,2021-12-15


## KPI 1: Total Watch Hours
**Formula:** SUM(watch_duration_min) / 60

In [6]:
# kpi 1
total_watch_hours = watch['watch_duration_min'].sum() / 60
print(f'Total Watch Hours: {total_watch_hours:,.0f}')

Total Watch Hours: 2,274,135


## KPI 2: Active Rate
**Formula:** Active Subscribers / Total Subscribers x 100  
**Target:** > 70%

In [7]:
# kpi 2
total_subs = len(subs)
active_subs = subs['is_active'].sum()

active_rate = active_subs / total_subs * 100
print(f'Total subscribers : {total_subs:,}')
print(f'Active subscribers: {active_subs:,}')
print(f'Active Rate       : {active_rate:.2f}%')

Total subscribers : 15,000
Active subscribers: 11,199
Active Rate       : 74.66%


## KPI 3: Churn Rate
**Formula:** Inactive Subscribers / Total Subscribers x 100  
**Target:** < 30%

In [8]:
# kpi 3
inactive_subs = total_subs - active_subs

churn_rate = inactive_subs / total_subs * 100
print(f'Inactive subscribers: {inactive_subs:,}')
print(f'Churn Rate          : {churn_rate:.2f}%')

Inactive subscribers: 3,801
Churn Rate          : 25.34%


## KPI 4: Average Completion Rate
**Formula:** AVG(completion_pct)  
**Target:** > 60%

In [9]:
# kpi 4
avg_completion = watch['completion_pct'].mean()
print(f'Avg Completion Rate: {avg_completion:.2f}%')

Avg Completion Rate: 65.33%


## KPI 5: Monthly Recurring Revenue (MRR)
**Formula:** SUM(monthly_price_usd) for active subscribers only

In [10]:
# kpi 5
active_only = subs[subs['is_active'] == True]

mrr = active_only['monthly_price_usd'].sum()
print(f'MRR: ${mrr:,.2f}')

MRR: $175,868.51


## KPI 6: ARPU (Average Revenue Per User)
**Formula:** MRR / Active Subscribers

In [11]:
# kpi 6
arpu = mrr / active_subs
print(f'ARPU: ${arpu:,.2f} per active subscriber per month')

ARPU: $15.70 per active subscriber per month


## KPI 7: Average Watch Time per Subscriber
**Formula:** Total Watch Hours / Active Subscribers (as given in the PRD)

In [12]:
# kpi 7
avg_watch_per_sub = total_watch_hours / active_subs
print(f'Avg Watch Time per Subscriber: {avg_watch_per_sub:,.1f} hours')

Avg Watch Time per Subscriber: 203.1 hours


## KPI 8: Watchlist Conversion
**Formula:** Watched / Watchlist Entries x 100  
**Target:** > 40%

In [13]:
# kpi 8
total_entries = len(watchlist)
watched_entries = watchlist['watched'].sum()

watchlist_conversion = watched_entries / total_entries * 100
print(f'Watchlist entries: {total_entries:,}')
print(f'Later watched    : {watched_entries:,}')
print(f'Watchlist Conversion: {watchlist_conversion:.2f}%')

Watchlist entries: 65,000
Later watched    : 30,048
Watchlist Conversion: 46.23%


## KPI 9: Hit Concentration
**Formula:** Plays from top 10% titles / Total plays x 100  


In [14]:
# kpi 9
# 1. Count plays (sessions) per title
plays_per_title = watch['title_id'].value_counts()

# 2. Include titles with zero plays, so '10% of titles' means 10% of the full catalogue
plays_per_title = plays_per_title.reindex(titles['title_id'], fill_value=0)

# 3. Sort high to low
plays_per_title = plays_per_title.sort_values(ascending=False)

# 4. Take the top 10% of titles
top_n = int(len(plays_per_title) * 0.10)
top_plays = plays_per_title.head(top_n).sum()

# 5. Share of all plays
hit_concentration = top_plays / plays_per_title.sum() * 100
print(f'Top 10% of titles = {top_n:,} titles')
print(f'Hit Concentration: {hit_concentration:.2f}% of all plays')

Top 10% of titles = 900 titles
Hit Concentration: 31.10% of all plays


## KPI 10: Originals Share of Hours
**Formula:** Hours on Originals / Total Hours x 100

In [15]:
# kpi 10
# Bring the 'is_original' flag from titles into watch history
watch_titles = pd.merge(watch, titles[['title_id', 'is_original']], on='title_id', how='left')

# Hours on Originals only
original_rows = watch_titles[watch_titles['is_original'] == True]
original_hours = original_rows['watch_duration_min'].sum() / 60

originals_share = original_hours / total_watch_hours * 100
print(f'Hours on Originals: {original_hours:,.0f}')
print(f'Originals Share of Hours: {originals_share:.2f}%')

Hours on Originals: 618,153
Originals Share of Hours: 27.18%


## KPI Summary Table
All 10 KPIs in one place, checked against the targets from the PRD.

In [16]:
def check(value, target, direction):
    # direction 'above' -> good if value > target; 'below' -> good if value < target
    if direction == 'above':
        return 'On target' if value > target else 'Below target'
    if direction == 'below':
        return 'On target' if value < target else 'Above target'
    return '-'

summary = pd.DataFrame({
    'KPI': [
        'Total Watch Hours', 'Active Rate (%)', 'Churn Rate (%)',
        'Avg Completion Rate (%)', 'MRR ($)', 'ARPU ($)',
        'Avg Watch Time / Subscriber (hrs)', 'Watchlist Conversion (%)',
        'Hit Concentration (%)', 'Originals Share of Hours (%)'
    ],
    'Value': [
        total_watch_hours, active_rate, churn_rate,
        avg_completion, mrr, arpu,
        avg_watch_per_sub, watchlist_conversion,
        hit_concentration, originals_share
    ],
    'Target': ['-', '> 70', '< 30', '> 60', '-', '-', '-', '> 40', '-', '-'],
    'Status': [
        '-',
        check(active_rate, 70, 'above'),
        check(churn_rate, 30, 'below'),
        check(avg_completion, 60, 'above'),
        '-', '-', '-',
        check(watchlist_conversion, 40, 'above'),
        '-', '-'
    ]
})

summary

,KPI,Value,Target,Status
0,Total Watch Hours,"2,274,135.13",-,-
1,Active Rate (%),74.66,> 70,On target
2,Churn Rate (%),25.34,< 30,On target
3,Avg Completion Rate (%),65.33,> 60,On target
4,MRR ($),"175,868.51",-,-
5,ARPU ($),15.70,-,-
6,Avg Watch Time / Subscriber (hrs),203.07,-,-
7,Watchlist Conversion (%),46.23,> 40,On target
8,Hit Concentration (%),31.10,-,-
9,Originals Share of Hours (%),27.18,-,-


---
## Bonus: Cohort Retention
Group subscribers by **signup month** (their cohort). For each cohort, find the % still active **3, 6 and 12 months** after signup.

Plan:
1. Cohort = signup month.
2. End date = churn date, or the snapshot date (2026-06-01) if still active.
3. Months lived = months between signup and end date.
4. Retained at N months = months lived >= N.
5. Only count a subscriber for N months if they signed up at least N months before the snapshot date (otherwise we cannot know yet).

In [17]:
snapshot = pd.Timestamp('2026-06-01')   # data is as of this date

cohort_df = subs.copy()

# 1. Cohort = signup month
cohort_df['cohort'] = cohort_df['signup_date'].dt.to_period('M')

# 2. End date: churn date if churned, otherwise snapshot date
cohort_df['end_date'] = cohort_df['churn_date'].fillna(snapshot)

# 3. Months lived (never below 0, in case of bad dates)
cohort_df['months_lived'] = ((cohort_df['end_date'].dt.year - cohort_df['signup_date'].dt.year) * 12
                             + (cohort_df['end_date'].dt.month - cohort_df['signup_date'].dt.month))
cohort_df['months_lived'] = cohort_df['months_lived'].clip(lower=0)

# 4 + 5. For each N, retained flag only for subscribers old enough
for n in [3, 6, 12]:
    old_enough = cohort_df['signup_date'] <= (snapshot - pd.DateOffset(months=n))
    retained = (cohort_df['months_lived'] >= n) * 1.0
    cohort_df[f'ret_{n}m'] = np.where(old_enough, retained, np.nan)

# Average per cohort, converted to %
cohort_table = cohort_df.groupby('cohort')[['ret_3m', 'ret_6m', 'ret_12m']].mean() * 100
cohort_table['subscribers'] = cohort_df.groupby('cohort').size()
cohort_table.columns = ['3-month %', '6-month %', '12-month %', 'Subscribers']

cohort_table.head(12)

,3-month %,6-month %,12-month %,Subscribers
cohort,,,,
2016-01,100.00,75.00,75.00,4
2016-02,88.89,88.89,88.89,9
2016-03,100.00,100.00,100.00,12
2016-04,100.00,100.00,100.00,13
2016-05,100.00,90.00,85.00,20
2016-06,100.00,100.00,95.24,21
2016-07,100.00,100.00,100.00,17
2016-08,100.00,100.00,100.00,24
2016-09,100.00,100.00,100.00,21


In [18]:
# Overall retention across ALL cohorts
print('Overall retention (all eligible subscribers):')
print(f"  After  3 months: {cohort_df['ret_3m'].mean() * 100:.2f}%")
print(f"  After  6 months: {cohort_df['ret_6m'].mean() * 100:.2f}%")
print(f"  After 12 months: {cohort_df['ret_12m'].mean() * 100:.2f}%")

Overall retention (all eligible subscribers):
  After  3 months: 98.28%
  After  6 months: 95.72%
  After 12 months: 92.46%


In [19]:
# Yearly view is easier to read than 100+ monthly rows
cohort_df['cohort_year'] = cohort_df['signup_date'].dt.year

yearly = cohort_df.groupby('cohort_year')[['ret_3m', 'ret_6m', 'ret_12m']].mean() * 100
yearly.columns = ['3-month %', '6-month %', '12-month %']
yearly

,3-month %,6-month %,12-month %
cohort_year,,,
2016,99.16,97.89,96.62
2017,99.65,99.12,97.36
2018,99.28,98.44,96.29
2019,99.24,97.91,96.02
2020,99.11,97.78,95.33
2021,99.31,97.54,94.44
2022,98.73,96.95,93.72
2023,99.07,96.50,91.62
2024,98.53,95.26,89.10
